<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/surfacewatercalc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math as mh
import pandas as pd
import numpy as np

def surfacewatercalc(lanes, Partable2, dailyflow, Treatment):
    # Initialize all return variables with default values
    Epump, Pump_area = 0, 0
    screen_headloss, screen_area_total, microsieves_headloss, microsieves_area = 0, 0, 0, 0
    Coag_area, Ecoag, coagheadloss, floc_area, Efloc, flocheadloss, coagtype,amountofcoagday = 0, 0, 0, 0, 0, 0, "Alum", 0
    clar_area, Eclar, clarhead, chemarearequired, Echem = 0, 0, 0, 0, 0
    Treatment,dailyflow, lanes,screen_area_total,Pump_area,,Epump,microsieves_area,Coag_area,Ecoag, floc_area, Efloc, clar_area, Eclar,rapidfilterprocessArea,
                                                     Etot,Eback,ompArea,Eomp,disarea,Edis,stor_area,softarea,softenergy,ufarea,ufenergy,nomenergy, nomarea,aerarea,aerenergy,condarea,condenergy, Etot, Eback, rapidsandhead = 0, 0, 0, 0
    ompArea, Eomp, Homp, = 0, 0, 0
    disarea, disstorage, Edis, dosageflow, dosageflow2,disheadloss,Cldaily,ammdaily = 0, 0, 0, 0, 0, 0, 0, 0
    softarea, softenergy,lime_dosage_kg_per_day = 0, 0, 0
    nomarea, nomenergy, nomchem,nomhead = 0, 0, 0, 0
    ufarea, ufenergy = 0, 0
    stor_area, stor_height = 0, 0

    # Get available processes from Treatment dataframe
    available_processes = Treatment['Treatment Steps'].unique()

    # Skip process selection if only one process is available
    if len(available_processes) == 1:
        print(f"Only one treatment process available: {available_processes[0]}")
        selected_processes = available_processes
    else:
        # Display process selection menu
        print("\n=== TREATMENT PROCESS SELECTION ===")
        print("The selected treatment processes are:")
        for i, process in enumerate(available_processes, 1):
            print(f"{i}. {process}")
        print("A. Calculate ALL processes")
        print("Q. Quit/Exit")

        while True:
            user_input = input("\nSelect processes to calculate (comma-separated numbers, 'A' for all, or 'Q' to quit): ").strip().upper()

            if user_input == 'A':
                selected_processes = available_processes
                break
            elif user_input == 'Q':
                print("Calculation cancelled by user.")
                return None
            else:
                try:
                    selected_indices = [int(i.strip()) for i in user_input.split(',')]
                    if all(1 <= i <= len(available_processes) for i in selected_indices):
                        selected_processes = [available_processes[i-1] for i in selected_indices]
                        break
                    else:
                        print(f"Please enter numbers between 1 and {len(available_processes)}")
                except ValueError:
                    print("Invalid input. Please enter numbers separated by commas, 'A' for all, or 'Q' to quit.")

        print("\nSelected processes:")
        for process in selected_processes:
            print(f"- {process}")

    # Flow selection
    if len(dailyflow) >= 2:
        try:
            designin = int(input("\nEnter design index flow (1 for the initial population etc.): "))
            index = 2 * designin - 1
            dailyflow_value = dailyflow[index]
        except (ValueError, IndexError):
            print("Invalid input or index out of range. Using last value in dailyflow.")
            dailyflow_value = dailyflow[-1]
    else:
        dailyflow_value = max(dailyflow)

    dailyflow = dailyflow_value

    # Process calculations based on selection
    if "Screens" in selected_processes and (Treatment['Treatment Steps'] == "Screens").any():
        %run "/content/drive/My Drive/Colab Notebooks/WP6/intake.ipynb"
        screen_headloss, screen_area_total, microsieves_headloss, microsieves_area = intake(Partable2, dailyflow, Treatment, lanes)

    if "Coagulation" in selected_processes and (Treatment['Treatment Steps'] == "Coagulation").any():
        %run "/content/drive/My Drive/Colab Notebooks/WP6/coagfunc.ipynb"
        Coag_area, Ecoag,coagheadloss, floc_area, Efloc, flocheadloss, clar_area, Eclar,clarhead,chemarearequired,coagtype,amountofcoagday = coagfunc(Partable2, dailyflow, Treatment, lanes)

    if any(p.lower().strip() == "sand filtration" for p in selected_processes) and (Treatment['Treatment Steps'].str.lower().str.strip() == "sand filtration").any():
        %run "/content/drive/My Drive/Colab Notebooks/WP6/rapidsandfilter.ipynb"
        rapidfilterprocessArea, Etot, Eback, rapidsandhead = rapidsandfilter(lanes, Partable2, dailyflow, Treatment)

    if any(p.lower().strip() == "nom removal" for p in selected_processes) and (Treatment['Treatment Steps'].str.lower().str.strip() == "nom removal").any():
        %run "/content/drive/My Drive/Colab Notebooks/WP6/nom_removal.ipynb"
        nomarea, nomenergy, nomchem,nomhead = nom_removal(lanes, Partable2, dailyflow, Treatment)

    if any(p.lower().strip() == "omp removal" for p in selected_processes) and (Treatment['Treatment Steps'].str.lower().str.strip() == "omp removal").any():
        %run "/content/drive/My Drive/Colab Notebooks/WP6/omp_removal.ipynb"
        ompArea, Eomp, Homp = omp_removal(lanes, Partable2, dailyflow, Treatment)

    if any(p.lower().strip() == "disinfection" for p in selected_processes) and (Treatment['Treatment Steps'].str.lower().str.strip() == "disinfection").any():
        %run "/content/drive/My Drive/Colab Notebooks/WP6/disinfection.ipynb"
        disarea,disstorage,Edis,dosageflow,dosageflow2,disheadloss,Cldaily,ammdaily = disin(lanes, Partable2, dailyflow, Treatment)

    if any(p.lower().strip() == "softening" for p in selected_processes) and (Treatment['Treatment Steps'].str.lower().str.strip() == "softening").any():
        %run "/content/drive/My Drive/Colab Notebooks/WP6/hardness.ipynb"
        softarea,softenergy,lime_dosage_kg_per_day = hardness(lanes, Partable2, dailyflow, Treatment)

    if "Ultrafiltration" in selected_processes and (Treatment['Treatment Steps'] == "Ultrafiltration").any():
        %run "/content/drive/My Drive/Colab Notebooks/WP6/ultrafiltration.ipynb"
        ufarea, ufenergy = ultrafiltration(lanes, Partable2, dailyflow, Treatment)

    response = input("Is a storage tank for the clean water required? (Yes/No): ")
    if response== "No":
        stor_height=0
        stor_area=0
    else:
        while True:
            stor_time = int(input("Set the storage tank retention time in hours (min 5 hours - max 72hours, default is 24): ") or 24)
            if 5 <= stor_time <= 72:
               break
            else:
               print("Invalid input. Enter a value between 5 and 72.")
        flowRate = dailyflow/24
        stor_volume = flowRate*stor_time
        if stor_volume< 10:
            stor_area = 5
            stor_height=2
        else:
           while True:
               stor_height = float(input("Set the depth of the clean water storage tank in meters(min 2m - max 5m - default 2.5m):") or 2.5)
               if 2 <= stor_height <= 5:
                   break
               else:
                  print("Invalid input. Enter a value between 2 and 5.")
           stor_area = stor_volume/stor_height
           stor_area = mh.ceil(stor_area)
        print(f'The clean water storage tank area is {stor_area}m2')
        new_row =pd.DataFrame([['Storage', 'Clean Water Storage Tank']], columns=['Treatment Steps', 'Treatment Technologies'])
        Treatment = pd.concat([Treatment, new_row], ignore_index=True)

    return  Treatment,Epump, Pump_area, screen_headloss, screen_area_total, microsieves_headloss, microsieves_area,\
            Coag_area, Ecoag,coagheadloss, floc_area, Efloc, flocheadloss, clar_area, Eclar,clarhead,chemarearequired,coagtype,amountofcoagday,\
            rapidfilterprocessArea, Etot, Eback, rapidsandhead, ompArea, Eomp, Homp,\
            disarea, disstorage, Edis, dosageflow, dosageflow2,disheadloss, Cldaily, ammdaily,stor_area, stor_height,\
            softarea, softenergy, lime_dosage_kg_per_day, ufarea, ufenergy, nomarea, nomenergy, nomchem,nomhead
